# Trier les triptyques

Ce notebook essaie de mettre en avant les triptyques clés :
- il prends d'abord les triptyques ou il y a un personnage en sujet ou objet
- puis il regarde s'il y a un verbe dynamouv

In [16]:
import pandas as pd

TRIPTYQUES_CSV = "../results/csv_triptyques/auto_all_chap_temps.csv"
DYNAMOUV_CSV = "../data/dynaMouv.csv"
OUT_CSV = "../results/csv_triptyques/triptyques_personnage_dynamouv.csv"

trip = pd.read_csv(TRIPTYQUES_CSV)
dyna = pd.read_csv(DYNAMOUV_CSV, sep=";")

In [17]:
verbes_dynamouv = set(dyna["Verbe"].dropna())

mask_personnage = trip["a_personnage_triptyque"]
mask_dynamouv = trip["Lemme_verbe"].fillna(trip["Verbe"]).isin(verbes_dynamouv)

#trip_filtre = trip[mask_personnage].copy()
trip_filtre = trip[mask_personnage & mask_dynamouv].copy()

trip_filtre = trip_filtre.sort_values(["chapter", "Num_paragr", "Num_phrase", "ID_verbe"],na_position="last")

trip_filtre.to_csv(OUT_CSV, index=False, encoding="utf-8")

print("Triptyques filtrés :", trip_filtre.shape)
trip_filtre.head(20)

Triptyques filtrés : (1254, 47)


,Phrase,Sujet,Verbe,Objet,Dep_sujet,Dep_verbe,Dep_objet,ID_sujet,ID_verbe,ID_objet,...,time_code,time_datetime,time_debut,time_fin,time_jour_romanesque,time_duration_value,time_duration_unit,time_duration_relation,time_source,time_ud_governor
43,À qui s' étonnerait de ce qu' un gentleman aus...,il,passa,sur la recommandation de MM. Baring frères,nsubj,ccomp,obl:arg,24.0,25,28.0,...,1872-00-00-00-00,1872,NaN,NaN,NaN,NaN,NaN,NaN,precedent_context,habitée
64,Avait -il voyagé?,-il,voyagé,NaN,nsubj,root,NaN,2.0,3,NaN,...,1872-00-00-00-00,1872,NaN,NaN,NaN,NaN,NaN,NaN,precedent_context,habitée
66,Il n' était endroit si reculé dont il ne parût...,il,parût,dont,nsubj,acl:relcl,iobj,8.0,10,7.0,...,1872-00-00-00-00,1872,NaN,NaN,NaN,NaN,NaN,NaN,precedent_context,habitée
71,"Quelquefois, mais en peu de mots, brefs et cla...",qui,circulaient,au sujet des voyageurs,nsubj,acl:relcl,obl:mod,18.0,19,24.0,...,1872-00-00-00-00,1872,NaN,NaN,NaN,NaN,NaN,NaN,precedent_context,habitée
80,C' était un homme qui avait dû voyager partout...,qui,voyager,NaN,nsubj,xcomp,NaN,5.0,8,NaN,...,1872-00-00-00-00,1872,NaN,NaN,NaN,NaN,NaN,NaN,precedent_context,habitée
82,"Ce qui était certain toutefois, c' est que, de...",Phileas Fogg,quitté,depuis de longues années,nsubj,ccomp,obl:mod,16.0,21,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,objet,quitté
83,"Ce qui était certain toutefois, c' est que, de...",Phileas Fogg,quitté,Londres,nsubj,ccomp,obj,16.0,21,22.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ud_direct,quitté
88,Ceux qui avaient l' honneur de le connaître un...,il,parcourait,qu',nsubj,acl:relcl,obj,27.0,28,26.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ud_direct,parcourait
89,Ceux qui avaient l' honneur de le connaître un...,il,parcourait,chaque jour,nsubj,acl:relcl,obl:mod,27.0,28,30.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,objet,parcourait
90,Ceux qui avaient l' honneur de le connaître un...,il,venir,de sa maison au club,nsubj,advcl,obl:arg,27.0,32,35.0,...,1872-00-00-00-00,1872,NaN,NaN,NaN,NaN,NaN,NaN,precedent_context,habitée


In [19]:
# Prépare une table dynaMouv réduite aux colonnes utiles
dyna_type = dyna[[
    "Verbe",
    "Macro-catégorie",
    "Catégorie de base",
    "Aspect lexical",
    "Manière"
]].copy()

# Renomme les colonnes pour que ce soit plus lisible
dyna_type = dyna_type.rename(columns={
    "Verbe": "Lemme_verbe",
    "Macro-catégorie": "macro_categorie_dynamouv",
    "Catégorie de base": "categorie_base_dynamouv",
    "Aspect lexical": "aspect_lexical_dynamouv",
    "Manière": "maniere_dynamouv"
})

# Ajoute les informations dynaMouv au fichier filtré
trip_presentation = trip_filtre.merge(
    dyna_type,
    on="Lemme_verbe",
    how="left"
)


# Colonnes lisibles à garder
colonnes_presentation = [
    "chapter",
    "Phrase",
    "Sujet",
    "Verbe",
    "Objet",
    "time_code",
    "lieux_tryptiques",
    "lemme_verbe_dynamouv",
    "macro_categorie_dynamouv",
    "categorie_base_dynamouv",
    "aspect_lexical_dynamouv",
    "maniere_dynamouv",
]

# Garde seulement les colonnes qui existent vraiment dans le fichier
colonnes_presentation = [
    c for c in colonnes_presentation
    if c in trip_presentation.columns
]

# Table finale plus lisible
trip_presentation = trip_presentation[colonnes_presentation].copy()

# Export
OUT_PRESENTATION_CSV = "../results/csv_triptyques/triptyques_personnage_dynamouv_presentation.csv"

trip_presentation.to_csv(OUT_PRESENTATION_CSV,index=False,encoding="utf-8")

print(f"dim : {trip_presentation.shape}")
trip_presentation.head(20)

dim : (1259, 10)


,chapter,Phrase,Sujet,Verbe,Objet,time_code,macro_categorie_dynamouv,categorie_base_dynamouv,aspect_lexical_dynamouv,maniere_dynamouv
0,1.0,À qui s' étonnerait de ce qu' un gentleman aus...,il,passa,sur la recommandation de MM. Baring frères,1872-00-00-00-00,Dpt au sens large,Dpt au sens strict,télique,NaN
1,1.0,Avait -il voyagé?,-il,voyagé,NaN,1872-00-00-00-00,Dpt au sens large,Dpt au sens faible,atélique,manière
2,1.0,Il n' était endroit si reculé dont il ne parût...,il,parût,dont,1872-00-00-00-00,Dpt au sens large,Dpt au sens strict,télique,NaN
3,1.0,"Quelquefois, mais en peu de mots, brefs et cla...",qui,circulaient,au sujet des voyageurs,1872-00-00-00-00,Dpt au sens large,Dpt au sens faible,atélique,manière
4,1.0,C' était un homme qui avait dû voyager partout...,qui,voyager,NaN,1872-00-00-00-00,Dpt au sens large,Dpt au sens faible,atélique,manière
5,1.0,"Ce qui était certain toutefois, c' est que, de...",Phileas Fogg,quitté,depuis de longues années,NaN,Dpt au sens large,Dpt au sens strict,télique,NaN
6,1.0,"Ce qui était certain toutefois, c' est que, de...",Phileas Fogg,quitté,Londres,NaN,Dpt au sens large,Dpt au sens strict,télique,NaN
7,1.0,Ceux qui avaient l' honneur de le connaître un...,il,parcourait,qu',NaN,Dpt au sens large,Dpt au sens faible,atélique,manière
8,1.0,Ceux qui avaient l' honneur de le connaître un...,il,parcourait,chaque jour,NaN,Dpt au sens large,Dpt au sens faible,atélique,manière
9,1.0,Ceux qui avaient l' honneur de le connaître un...,il,venir,de sa maison au club,1872-00-00-00-00,Dpt au sens large,Dpt au sens strict,télique,NaN
